# Table Access Protocol (TAP) queries
Ref: [TAP](https://dp1.lsst.io/tutorials/notebook/102/notebook-102-1.html)

In [1]:
import numpy as np
from lsst.rsp import get_tap_service
import pandas as pd

service = get_tap_service ('tap')
assert service is not None

fields = pd.DataFrame(columns=['ra_deg', 'dec_deg'])
fields.loc['47Tuc']          = [  6.02, -72.08] # 47 Tuc Globular 
fields.loc['LELF_SV_38_7']   = [ 37.86,   6.98] # 
fields.loc['Fornax']         = [ 40.00, -34.45] # 
fields.loc['ECDFS']          = [ 53.13, -28.10] # 
fields.loc['EDFS']           = [ 59.10, -48.73] # 
fields.loc['LGLF_SV_95_-25'] = [ 95.00, -25.00] # 
fields.loc['Seagull_Nebula'] = [106.23, -10.51] # 

ddeg = .01
def grabcone(field):
    return "CONTAINS(POINT('ICRS', coord_ra, coord_dec), " +\
           f"CIRCLE('ICRS', {fields.loc[field, 'ra_deg']}, " +\
                          f"{fields.loc[field, 'dec_deg']}, {ddeg})) = 1"
def grabpoint(field):
    return f"POINT('ICRS', {fields.loc[field, 'ra_deg']}, " +\
                         f"{fields.loc[field, 'dec_deg']})"

In [2]:
field = 'ECDFS'
#                               COLUMNS  CATALOG CONSTRAINTS
queries = pd.DataFrame(columns=['select', 'from', 'where', 'order by'])
# queries.loc['tap_schemas'] = ["*", "tap_schema", "", ""]          Table [ tap_schema ] is not found in TapSchema. Possible reasons: table does not exist or permission is denied.
queries.loc['schemas'] = ["*", "tap_schema.schemas", "", ""]          # list all schemas
queries.loc['tables'] =  ["*",                                        # list all table names from DP1 TAP schema
                          "tap_schema.tables", 
                          "tap_schema.tables.schema_name = 'dp1'", 
                          "table_index ASC"]
queries.loc['columns'] = ["column_name, datatype, description, unit", # list all columns from DP1 Object table
                          "tap_schema.columns",
                          "table_name = 'dp1.Object'", ""]
queries.loc['cone']    = ["coord_ra, coord_dec", 
                          "dp1.Object", grabcone(field), ""]
queries.loc['sort']    = ["coord_ra, coord_dec", 
                          "dp1.Object", grabcone(field), "coord_ra ASC"]
queries.loc['top']     = ["TOP 10 coord_ra, coord_dec", 
                          "dp1.Object", grabcone(field), "coord_ra ASC"]
queries.loc['rename']  = ["coord_ra AS ra_deg, coord_dec AS dec_deg", 
                          "dp1.Object", grabcone(field), ""]
queries.loc['deg2rad'] = ["RADIANS(coord_ra) AS ra_radians, RADIANS(coord_dec) AS dec_radians", 
                          "dp1.Object", grabcone(field), ""]
queries.loc['dstance'] = ["coord_ra, coord_dec, " +
                           "DISTANCE(POINT('ICRS', coord_ra, coord_dec), " + grabpoint(field) + ")",
                          "dp1.Object", grabcone(field), ""]                           
queries.loc['mag']     = ["coord_ra, coord_dec, " +
                          "u_cModelMag, g_cModelMag, r_cModelMag, i_cModelMag, z_cModelMag, y_cModelMag",
                          "dp1.Object", grabcone(field), ""]
queries.loc['relat']   = ["coord_ra, coord_dec, " +
                          "u_cModelMag, g_cModelMag, r_cModelMag, i_cModelMag, z_cModelMag, y_cModelMag",
                          "dp1.Object", 
                          grabcone(field) + " AND u_cModelMag < 25 AND g_cModelMag < 25"
                                         + " AND r_cModelMag < 25 AND i_cModelMag < 25"
                                         + " AND z_cModelMag < 25 AND y_cModelMag < 25", ""]
queries.loc['btween']  = ["objectId, u_psfMAG, g_psfMag, r_psfMag, i_psfMag, z_psfMag, y_psfMag", 
                          "dp1.Object", 
                          grabcone(field) + " AND r_psfMag BETWEEN g_psfMag AND i_psfMag", ""]
queries.loc['in']      = ["objectId, coord_ra, coord_dec",
                          "dp1.Object", 
                          "objectID IN ({})".format(", ".join(str(dd) for dd in [611255141361789816, 611255141361790690, 611255141361791996])), ""]
queries.loc['jy2ab']   = ["coord_ra, coord_dec, " +
                          "scisql_nanojanskyToAbMag(u_sersicFlux) AS u_sersicMag, " +
                          "scisql_nanojanskyToAbMag(g_sersicFlux) AS g_sersicMag, " +
                          "scisql_nanojanskyToAbMag(r_sersicFlux) AS r_sersicMag, " +
                          "scisql_nanojanskyToAbMag(i_sersicFlux) AS i_sersicMag, " +
                          "scisql_nanojanskyToAbMag(z_sersicFlux) AS z_sersicMag, " +
                          "scisql_nanojanskyToAbMag(y_sersicFlux) AS y_sersicMag",
                          "dp1.Object", grabcone(field), ""]
queries.loc['sigma']   = ["coord_ra, coord_dec, " +
                          "scisql_nanojanskyToAbMag(g_sersicFlux) AS g_sersicMag, " +
                          "scisql_nanojanskyToAbMagSigma(g_sersicFlux, g_sersicFluxErr) AS g_sersicMagErr",
                          "dp1.Object", grabcone(field), ""]
queries.loc['dif2clr'] = ["coord_ra, coord_dec, " +
                          "u_cModelMag - g_cModelMag AS ug_clr, " +
                          "g_cModelMag - r_cModelMag AS gr_clr, " +
                          "r_cModelMag - i_cModelMag AS ri_clr, " +
                          "i_cModelMag - z_cModelMag AS iz_clr, " +
                          "z_cModelMag - y_cModelMag AS zy_clr",
                          "dp1.Object", grabcone(field), ""]
queries

,select,from,where,order by
schemas,*,tap_schema.schemas,,
tables,*,tap_schema.tables,tap_schema.tables.schema_name = 'dp1',table_index ASC
columns,"column_name, datatype, description, unit",tap_schema.columns,table_name = 'dp1.Object',
cone,"coord_ra, coord_dec",dp1.Object,"CONTAINS(POINT('ICRS', coord_ra, coord_dec), C...",
sort,"coord_ra, coord_dec",dp1.Object,"CONTAINS(POINT('ICRS', coord_ra, coord_dec), C...",coord_ra ASC
top,"TOP 10 coord_ra, coord_dec",dp1.Object,"CONTAINS(POINT('ICRS', coord_ra, coord_dec), C...",coord_ra ASC
rename,"coord_ra AS ra_deg, coord_dec AS dec_deg",dp1.Object,"CONTAINS(POINT('ICRS', coord_ra, coord_dec), C...",
deg2rad,"RADIANS(coord_ra) AS ra_radians, RADIANS(coord...",dp1.Object,"CONTAINS(POINT('ICRS', coord_ra, coord_dec), C...",
dstance,"coord_ra, coord_dec, DISTANCE(POINT('ICRS', co...",dp1.Object,"CONTAINS(POINT('ICRS', coord_ra, coord_dec), C...",
mag,"coord_ra, coord_dec, u_cModelMag, g_cModelMag,...",dp1.Object,"CONTAINS(POINT('ICRS', coord_ra, coord_dec), C...",


In [3]:
cols = queries.columns
def grabout(tisq):
    for idx, dat in tisq.iterrows():
        query = ''
        for col in cols:
            if dat[col] != '':
                query += f' {col.upper()} {dat[col]}'
        print(idx); print(query)
        if 'schema' not in dat['from']:
            job = service.submit_job(query)
            job.run()
            job.wait(phases=['COMPLETED', 'ERROR'])
            if job.phase == 'ERROR':
                job.raise_if_error()
            elif job.phase == 'COMPLETED':
                out = job.fetch_result().to_table()
        else:
            out = service.search(query).to_table()
        tisq.loc[idx, 'out'] = out
        print(type(out), len(out))
#       display(out)
        print()
grabout(queries)

schemas
 SELECT * FROM tap_schema.schemas
<class 'astropy.table.table.Table'> 4

tables
 SELECT * FROM tap_schema.tables WHERE tap_schema.tables.schema_name = 'dp1' ORDER BY table_index ASC
<class 'astropy.table.table.Table'> 12

columns
 SELECT column_name, datatype, description, unit FROM tap_schema.columns WHERE table_name = 'dp1.Object'
<class 'astropy.table.table.Table'> 1296

cone
 SELECT coord_ra, coord_dec FROM dp1.Object WHERE CONTAINS(POINT('ICRS', coord_ra, coord_dec), CIRCLE('ICRS', 53.13, -28.1, 0.01)) = 1
<class 'astropy.table.table.Table'> 136

sort
 SELECT coord_ra, coord_dec FROM dp1.Object WHERE CONTAINS(POINT('ICRS', coord_ra, coord_dec), CIRCLE('ICRS', 53.13, -28.1, 0.01)) = 1 ORDER BY coord_ra ASC
<class 'astropy.table.table.Table'> 136

top
 SELECT TOP 10 coord_ra, coord_dec FROM dp1.Object WHERE CONTAINS(POINT('ICRS', coord_ra, coord_dec), CIRCLE('ICRS', 53.13, -28.1, 0.01)) = 1 ORDER BY coord_ra ASC
<class 'astropy.table.table.Table'> 10

rename
 SELECT coord_ra

        DISTANCE(POINT('ICRS', coord_ra, coord_dec), 
        POINT('ICRS', 6.02, -72.08)) 
        FROM dp1.Object 
        WHERE CONTAINS(POINT('ICRS', coord_ra, coord_dec), 
        CIRCLE('ICRS', 6.02, -72.08, 0.01)) = 1

        DISTANCE(POINT('ICRS', coord_ra, coord_dec),
        POINT('ICRS', 53, -28)) AS distance
        FROM dp1.Object
        WHERE CONTAINS(POINT('ICRS', coord_ra, coord_dec),
        CIRCLE('ICRS', 53, -28, 0.01)) = 1

In [4]:
queries

,select,from,where,order by,out
schemas,*,tap_schema.schemas,,,"[[dp02_dc2_catalogs, , Data Preview 0.2 contai..."
tables,*,tap_schema.tables,tap_schema.tables.schema_name = 'dp1',table_index ASC,"[[dp1, dp1.Object, table, , Descriptions of st..."
columns,"column_name, datatype, description, unit",tap_schema.columns,table_name = 'dp1.Object',,"[[coord_dec, double, Fiducial ICRS Declination..."
cone,"coord_ra, coord_dec",dp1.Object,"CONTAINS(POINT('ICRS', coord_ra, coord_dec), C...",,"[[53.12501957694767, -28.108550690747236], [53..."
sort,"coord_ra, coord_dec",dp1.Object,"CONTAINS(POINT('ICRS', coord_ra, coord_dec), C...",coord_ra ASC,"[[53.11912319968864, -28.09869799399022], [53...."
top,"TOP 10 coord_ra, coord_dec",dp1.Object,"CONTAINS(POINT('ICRS', coord_ra, coord_dec), C...",coord_ra ASC,"[[53.11912319968864, -28.09869799399022], [53...."
rename,"coord_ra AS ra_deg, coord_dec AS dec_deg",dp1.Object,"CONTAINS(POINT('ICRS', coord_ra, coord_dec), C...",,"[[53.12501957694767, -28.108550690747236], [53..."
deg2rad,"RADIANS(coord_ra) AS ra_radians, RADIANS(coord...",dp1.Object,"CONTAINS(POINT('ICRS', coord_ra, coord_dec), C...",,"[[0.9272065068041819, -0.4905867575172657], [0..."
dstance,"coord_ra, coord_dec, DISTANCE(POINT('ICRS', co...",dp1.Object,"CONTAINS(POINT('ICRS', coord_ra, coord_dec), C...",,"[[53.12501957694767, -28.108550690747236, 0.00..."
mag,"coord_ra, coord_dec, u_cModelMag, g_cModelMag,...",dp1.Object,"CONTAINS(POINT('ICRS', coord_ra, coord_dec), C...",,"[[53.12501957694767, -28.108550690747236, 26.3..."


In [5]:
# list all schemas
queries.loc['schemas', 'out']
# 2 out of 4 are relevant to me: dp1, tap_schema

schema_name,utype,description,schema_index
str64,str512,str512,int32
dp02_dc2_catalogs,,"Data Preview 0.2 contains the image and catalog products of the Rubin Science Pipelines v23 processing of the DESC Data Challenge 2 simulation, which covered 300 square degrees of the wide-fast-deep LSST survey region over 5 years.",2
dp1,,"Data Preview 1 contains image and catalog products from the Rubin Science Pipelines v29 processing of observations obtained with the LSST Commissioning Camera of seven ~1 square degree fields, over seven weeks in late 2024.",0
ivoa,,ObsCore v1.1 attributes in ObsTAP realization,1
tap_schema,,A TAP-standard-mandated schema to describe tablesets in a TAP 1.1 service,100000


In [6]:
# list all table names from DP1 TAP schema
queries.loc['tables', 'out']
# column 'table_name' matches the list of 12 tables under Schema Browser > Data Preview 1 
# https://sdm-schemas.lsst.io/dp1.html

schema_name,table_name,table_type,utype,description,table_index
str512,str64,str8,str512,str512,int32
dp1,dp1.Object,table,,Descriptions of static astronomical objects (or the static aspects of variable and slowly-moving objects) detected and measured on coadds.,10
dp1,dp1.Source,table,,"Properties of detections on the single-epoch visit images, performed independently of the Object detections on coadded images.",20
dp1,dp1.ForcedSource,table,,"Forced-photometry measurements on individual single-epoch visit images and difference images, based on and linked to the entries in the Object table. Point-source PSF photometry is performed, based on coordinates from a reference band chosen for each Object and reported in the Object.refBand column.",30
dp1,dp1.DiaObject,table,,Properties of time-varying astronomical objects based on association of data from one or more spatially-related DiaSource detections on individual single-epoch difference images.,40
dp1,dp1.DiaSource,table,,Properties of transient-object detections on the single-epoch difference images.,50
dp1,dp1.ForcedSourceOnDiaObject,table,,"Point-source forced-photometry measurements on individual single-epoch visit images and difference images, based on and linked to the entries in the DiaObject table.",60
dp1,dp1.CoaddPatches,table,,Static information about the subset of tracts and patches from the standard LSST skymap that apply to coadds in these catalogs,70
dp1,dp1.Visit,table,,"Metadata about the pointings of the telescope, largely associated with the boresight of the LSSTComCam focal plane as a whole.",80
dp1,dp1.CcdVisit,table,,Metadata about the nine individual CCD images for each Visit in the DP1 LSSTComCam dataset.,90


In [7]:
# list all columns from DP1 Object table
queries.loc['columns', 'out'].to_pandas()
# same as the 1296 rows in ./DP1_schema/Object.csv

,column_name,datatype,description,unit
0,coord_dec,double,Fiducial ICRS Declination of centroid used for...,deg
1,coord_decErr,float,Error in fiducial ICRS Declination of centroid,deg
2,coord_ra,double,Fiducial ICRS Right Ascension of centroid used...,deg
3,coord_ra_dec_Cov,float,Covariance between fiducial ICRS Right Ascensi...,deg**2
4,coord_raErr,float,Error in fiducial ICRS Right Ascension of cent...,deg
...,...,...,...,...
1291,z_raErr,float,"Error in right ascension, measured on z-band.",deg
1292,z_sersicFlux,float,z-band flux from the multiband Sersic model fit.,nJy
1293,z_sersicFluxErr,float,Error on the z-band flux from the multiband Se...,nJy
1294,z_sizeExtendedness,float,Moments-based measure of whether an object is ...,
